In [1]:
print('hello')

hello


# NB-02 — Cleaning & ISO-3 Standardisation
## The Peacekeepers' Arms Race: Testing the Stability-Instability Paradox

All eleven raw checkpoints from NB-01 are cleaned here and written to `data/clean/` as Parquet.
Every output file uses an `iso3` column (ISO 3166-1 alpha-3) as the common country key that
enables cross-source joins in downstream notebooks.

| Section | Task |
|---------|------|
| 0 | Load checkpoints, instantiate resolver |
| 1 | Resolver smoke tests |
| 2 | COW stateabb → ISO3 mapping |
| 3.1–3.10 | Clean each source individually |
| 4 | Build UCDP participation panel + `is_extraterritorial` flag |
| 5 | Per-source QA printouts |
| 6 | Cross-source coverage audit |
| 7 | Save unmatched audit |


---
## Section 0 — Setup

Load all 11 NB-01 checkpoints into memory and instantiate the ISO3Resolver.
The resolver is a shared instance used throughout the notebook; per-source
`dataset_overrides` dicts are passed where SIPRI/V-Dem naming quirks require it.


In [2]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path

from src.config import CHECKPOINT_DIR, CLEAN_DIR
from src.io_utils import load_checkpoint, save_checkpoint
from src.iso3 import ISO3Resolver, COW_TO_ISO3, gw_to_iso3_set

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# ── Load all NB01 checkpoints ─────────────────────────────────────────────────
ucdp_acd          = load_checkpoint(CHECKPOINT_DIR / "ucdp_acd_raw.parquet")
ucdp_ged          = load_checkpoint(CHECKPOINT_DIR / "ucdp_ged_raw.parquet")
ucdp_brd          = load_checkpoint(CHECKPOINT_DIR / "ucdp_brd_raw.parquet")
ucdp_dyadic       = load_checkpoint(CHECKPOINT_DIR / "ucdp_dyadic_raw.parquet")
sipri_milex       = load_checkpoint(CHECKPOINT_DIR / "sipri_milex_long_raw.parquet")
sipri_tiv_reg     = load_checkpoint(CHECKPOINT_DIR / "sipri_tiv_register_raw.parquet")
worldbank_wdi     = load_checkpoint(CHECKPOINT_DIR / "worldbank_wdi_raw.parquet")
vdem              = load_checkpoint(CHECKPOINT_DIR / "vdem_raw.parquet")
cow_cinc          = load_checkpoint(CHECKPOINT_DIR / "cow_cinc_raw.parquet")

# ── Instantiate resolver ──────────────────────────────────────────────────────
resolver = ISO3Resolver()

# ── Row count summary ─────────────────────────────────────────────────────────
_sources = {
    "ucdp_acd":          ucdp_acd,
    "ucdp_ged":          ucdp_ged,
    "ucdp_brd":          ucdp_brd,
    "ucdp_dyadic":       ucdp_dyadic,
    "sipri_milex":       sipri_milex,
    "sipri_tiv_register": sipri_tiv_reg,
    "worldbank_wdi":     worldbank_wdi,
    "vdem":              vdem,
    "cow_cinc":          cow_cinc,
}
print(f"{'Source':<25}  {'Rows':>8}")
print("-" * 36)
for name, df in _sources.items():
    print(f"{name:<25}  {len(df):>8,}")

[checkpoint] loaded ← ucdp_acd_raw.parquet  (2,752 rows)
[checkpoint] loaded ← ucdp_ged_raw.parquet  (385,918 rows)
[checkpoint] loaded ← ucdp_brd_raw.parquet  (1,586 rows)
[checkpoint] loaded ← ucdp_dyadic_raw.parquet  (3,432 rows)
[checkpoint] loaded ← sipri_milex_long_raw.parquet  (8,435 rows)
[checkpoint] loaded ← sipri_tiv_register_raw.parquet  (60,789 rows)
[checkpoint] loaded ← worldbank_wdi_raw.parquet  (17,195 rows)
[checkpoint] loaded ← vdem_raw.parquet  (13,080 rows)
[checkpoint] loaded ← cow_cinc_raw.parquet  (15,951 rows)
Source                         Rows
------------------------------------
ucdp_acd                      2,752
ucdp_ged                    385,918
ucdp_brd                      1,586
ucdp_dyadic                   3,432
sipri_milex                   8,435
sipri_tiv_register           60,789
worldbank_wdi                17,195
vdem                         13,080
cow_cinc                     15,951


---
## Section 1 — Resolver Smoke Tests

Verify ~17 known name→ISO3 mappings before touching real data.
A failure here means `GLOBAL_OVERRIDES` needs updating before continuing.


In [3]:
SMOKE_TESTS = [
    ("United States",                    "USA"),
    ("Russia",                           "RUS"),
    ("USSR",                             "RUS"),
    ("Yugoslavia",                       "SRB"),
    ("Czechoslovakia",                   "CZE"),
    ("Burma",                            "MMR"),
    ("Zaire",                            "COD"),
    ("DR Congo (Zaire)",                 "COD"),
    ("Korea, South",                     "KOR"),
    ("Taiwan",                           "TWN"),
    ("Kosovo",                           "XKX"),
    ("Palestine",                        "PSE"),
    ("Timor Leste",                      "TLS"),
    ("Kyrgyz Republic",                  "KGZ"),
    ("German Democratic Republic",       "DEU"),
    ("Gambia, The",                      "GMB"),
    ("Hyderabad",                        None),
]

all_pass = True
for name, expected in SMOKE_TESTS:
    got = resolver.resolve(name, dataset="smoke_test")
    ok = got == expected
    if not ok:
        all_pass = False
    print(f"  {'PASS' if ok else 'FAIL'}  {name!r:<42} → {str(got):<6}  (expected {expected!r})")

assert all_pass, "Smoke tests failed — fix GLOBAL_OVERRIDES before continuing."
print("\nAll smoke tests passed.")


  PASS  'United States'                            → USA     (expected 'USA')
  PASS  'Russia'                                   → RUS     (expected 'RUS')
  PASS  'USSR'                                     → RUS     (expected 'RUS')
  PASS  'Yugoslavia'                               → SRB     (expected 'SRB')
  PASS  'Czechoslovakia'                           → CZE     (expected 'CZE')
  PASS  'Burma'                                    → MMR     (expected 'MMR')
  PASS  'Zaire'                                    → COD     (expected 'COD')
  PASS  'DR Congo (Zaire)'                         → COD     (expected 'COD')
  PASS  'Korea, South'                             → KOR     (expected 'KOR')
  PASS  'Taiwan'                                   → TWN     (expected 'TWN')
  PASS  'Kosovo'                                   → XKX     (expected 'XKX')
  PASS  'Palestine'                                → PSE     (expected 'PSE')
  PASS  'Timor Leste'                              → TLS     (ex

---
## Section 2 — COW Stateabb Mapping

COW NMC v6.0 uses its own 3-letter abbreviation system that differs from ISO 3166-1
alpha-3 in ~100 cases (e.g. `UKG`→`GBR`, `FRN`→`FRA`, `AUL`→`AUS`).

The `COW_TO_ISO3` dict in `src/iso3.py` covers all codes present in NMC-6.
Any remaining codes that happen to be valid ISO3 are passed through the general resolver.

CINC data ends at 2016 in NMC v6.0 — rows after 2016 are stray/partial records and dropped.


In [4]:
cow = cow_cinc.copy()
cow = cow[cow["year"] <= 2016].copy()

# Primary lookup: COW_TO_ISO3
cow["iso3"] = cow["stateabb"].map(COW_TO_ISO3)

# Fallback for any stateabb not in the dict (may already be valid ISO3)
fallback_mask = cow["iso3"].isna() & cow["stateabb"].notna()
cow.loc[fallback_mask, "iso3"] = cow.loc[fallback_mask, "stateabb"].apply(
    lambda x: resolver.resolve(x, dataset="cow_cinc")
)

cow["year"] = cow["year"].astype(int)

n_none = cow["iso3"].isna().sum()
print(f"COW CINC: {len(cow_cinc):,} raw → {len(cow):,} after ≤2016 filter")
print(f"  iso3 resolved: {(~cow['iso3'].isna()).sum():,}  |  unresolved (→dropped): {n_none:,}")
if n_none > 0:
    print("  Unresolved stateabb codes:")
    unres = cow[cow["iso3"].isna()]["stateabb"].value_counts()
    for code, cnt in unres.items():
        print(f"    {code}: {cnt} rows — add to COW_TO_ISO3 if needed")


COW CINC: 15,951 raw → 15,951 after ≤2016 filter
  iso3 resolved: 15,393  |  unresolved (→dropped): 558
  Unresolved stateabb codes:
    BAD: 56 rows — add to COW_TO_ISO3 if needed
    BAV: 56 rows — add to COW_TO_ISO3 if needed
    WRT: 56 rows — add to COW_TO_ISO3 if needed
    HSG: 52 rows — add to COW_TO_ISO3 if needed
    SAX: 52 rows — add to COW_TO_ISO3 if needed
    HSE: 51 rows — add to COW_TO_ISO3 if needed
    SIC: 46 rows — add to COW_TO_ISO3 if needed
    PAP: 45 rows — add to COW_TO_ISO3 if needed
    TUS: 45 rows — add to COW_TO_ISO3 if needed
    HAN: 30 rows — add to COW_TO_ISO3 if needed
    MEC: 25 rows — add to COW_TO_ISO3 if needed
    SNM: 25 rows — add to COW_TO_ISO3 if needed
    MOD: 19 rows — add to COW_TO_ISO3 if needed


---
## Section 3 — Clean Each Source

One subsection per source. Each subsection:
1. Applies ISO3 resolution via `resolver.resolve_series()`
2. Ensures `year` is `int` dtype
3. Writes a `*_clean.parquet` to `data/clean/`


### 3.1 UCDP Armed Conflict Dataset (ACD)

The `location` column contains the country/countries where fighting occurred.
Multi-country strings like `"India, Pakistan"` are split and exploded so that
each row represents one conflict-year-location tuple. `is_multi_location=True`
flags rows that originated from a multi-country string.


In [5]:
def _clean_acd(df, res):
    df = df.copy()
    df["is_multi_location"] = df["location"].str.contains(",", na=False)
    # Split on comma and strip each part
    df["_loc_list"] = df["location"].str.split(",").apply(
        lambda parts: [p.strip() for p in parts]
    )
    df = df.explode("_loc_list").reset_index(drop=True)
    df["iso3"] = res.resolve_series(df["_loc_list"], dataset="ucdp_acd")
    df = df.drop(columns=["_loc_list"])
    df["year"] = df["year"].astype(int)
    return df

ucdp_acd_clean = _clean_acd(ucdp_acd, resolver)
n_multi = ucdp_acd_clean["is_multi_location"].sum()
n_iso3  = ucdp_acd_clean["iso3"].notna().sum()
print(f"UCDP ACD: {len(ucdp_acd):,} rows → {len(ucdp_acd_clean):,} after explode")
print(f"  multi-location rows: {n_multi:,}  |  iso3 resolved: {n_iso3:,}")
save_checkpoint(ucdp_acd_clean, CLEAN_DIR / "ucdp_acd_clean.parquet")


UCDP ACD: 2,752 rows → 2,917 after explode
  multi-location rows: 316  |  iso3 resolved: 2,914
[checkpoint] saved → ucdp_acd_clean.parquet  (2,917 rows)


### 3.2 UCDP Georeferenced Event Dataset (GED)

`date_start` is parsed to datetime and a `year_month` period column is derived
(e.g. `"2010-03"`) for RQ2 monthly conflict-intensity aggregation.
The `country` column is the primary geographic key.


In [6]:
def _clean_ged(df, res):
    df = df.copy()
    df["iso3"] = res.resolve_series(df["country"], dataset="ucdp_ged")
    df["date_start"] = pd.to_datetime(df["date_start"], errors="coerce")
    df["year"] = df["year"].astype(int)
    df["year_month"] = df["date_start"].dt.to_period("M").astype(str)
    return df

ucdp_ged_clean = _clean_ged(ucdp_ged, resolver)
print(f"UCDP GED: {len(ucdp_ged_clean):,} rows")
print(f"  iso3 resolved: {ucdp_ged_clean['iso3'].notna().sum():,}")
print(f"  year_month sample: {ucdp_ged_clean['year_month'].dropna().iloc[0]}")
save_checkpoint(ucdp_ged_clean, CLEAN_DIR / "ucdp_ged_clean.parquet")


UCDP GED: 385,918 rows
  iso3 resolved: 385,918
  year_month sample: 2017-07
[checkpoint] saved → ucdp_ged_clean.parquet  (385,918 rows)


### 3.3 UCDP Battle-Related Deaths (BRD)

The `location_inc` column (location of the conflict dyad) is resolved to ISO3.
`bd_best`, `bd_low`, `bd_high` are kept as the primary death-estimate variables.


In [7]:
def _clean_brd(df, res):
    df = df.copy()
    df["iso3"] = res.resolve_series(df["location_inc"], dataset="ucdp_brd")
    df["year"] = df["year"].astype(int)
    return df

ucdp_brd_clean = _clean_brd(ucdp_brd, resolver)
print(f"UCDP BRD: {len(ucdp_brd_clean):,} rows")
print(f"  iso3 resolved: {ucdp_brd_clean['iso3'].notna().sum():,}")
save_checkpoint(ucdp_brd_clean, CLEAN_DIR / "ucdp_brd_clean.parquet")


UCDP BRD: 1,586 rows
  iso3 resolved: 1,536
[checkpoint] saved → ucdp_brd_clean.parquet  (1,586 rows)


### 3.4 UCDP Dyadic Dataset

The dyadic dataset disaggregates conflicts by side. `side_a` and `side_b` strings
may start with `"Government of "` (state actors) or be armed-group names (non-state).

`_resolve_ucdp_actor()` handles both cases and returns `(list[iso3], is_state)`.
Unit tests run below before touching the real data.


In [8]:
def _resolve_ucdp_actor(actor_string, res, dataset="ucdp_dyadic"):
    """Parse a UCDP actor string → (list[str|None], has_state: bool).

    - Strips 'Government of ' prefix (marks as state actor)
    - Splits comma-separated multi-actor strings
    - Non-state actors that fail resolution → None in the list
    - is_state = True if ANY actor resolves to a valid ISO3
    """
    if not isinstance(actor_string, str) or not actor_string.strip():
        return [], False

    tokens = [t.strip() for t in actor_string.split(",")]
    iso3_list = []
    is_state = False

    for token in tokens:
        if not token:
            continue
        is_govt = token.startswith("Government of ")
        clean = token[len("Government of "):].strip() if is_govt else token
        iso3 = res.resolve(clean, dataset=dataset)
        iso3_list.append(iso3)
        if iso3 is not None:
            is_state = True

    return iso3_list, is_state


# ── Unit tests — _resolve_ucdp_actor ─────────────────────────────────────────
_r = ISO3Resolver()

# 1. Government of prefix stripped correctly
_iso3s, _state = _resolve_ucdp_actor("Government of Ethiopia", _r)
assert _iso3s == ["ETH"] and _state is True, f"Test 1 failed: {_iso3s}"

# 2. Non-state armed group → None, is_state=False
_iso3s, _state = _resolve_ucdp_actor("IGLF", _r)
assert _iso3s == [None] and _state is False, f"Test 2 failed: {_iso3s}"

# 3. Multi-actor string (comma-separated governments)
_iso3s, _state = _resolve_ucdp_actor("Government of India, Government of Pakistan", _r)
assert "IND" in _iso3s and "PAK" in _iso3s and _state is True, f"Test 3 failed: {_iso3s}"

# 4. None / empty input
_iso3s, _state = _resolve_ucdp_actor(None, _r)
assert _iso3s == [] and _state is False, "Test 4 failed"

# 5. Government of with UCDP historical name
_iso3s, _state = _resolve_ucdp_actor("Government of DR Congo (Zaire)", _r)
assert _iso3s == ["COD"] and _state is True, f"Test 5 failed: {_iso3s}"

# 6. Already-resolved ISO3 name (direct state name, no 'Government of')
_iso3s, _state = _resolve_ucdp_actor("Russia", _r)
assert "RUS" in _iso3s and _state is True, f"Test 6 failed: {_iso3s}"

print("All _resolve_ucdp_actor unit tests passed.")


# ── _conflict_location_iso3_set — routing by type_of_conflict ─────────────────

def _conflict_location_iso3_set(row, res):
    """Resolve the conflict's territorial location to a set of ISO3 codes.

    Routing logic (by type_of_conflict):

    Types 3 & 4 (intrastate / internationalised intrastate):
        side_a is the defending government by UCDP convention → resolve to ISO3.
        Avoids gwno_loc coalition-encoding problem (e.g. Iraq 2003 type-4 conflicts
        where intervening states' GW codes are folded into gwno_loc).

    Type 2 (interstate):
        CASE A — single-state side_a (bilateral war, e.g. Kargil 1999):
            gwno_loc is reliable; use gw_to_iso3_set(gwno_loc).
        CASE B — multi-state side_a (coalition invasion, e.g. Iraq War 2003):
            UCDP folds all coalition GW codes into gwno_loc, making coalition
            members appear non-extraterritorial. Use side_b (the invaded country)
            as the conflict territory instead.

    Type 1 (extrasystemic):
        gwno_loc is reliable; use gw_to_iso3_set(gwno_loc).

    Fallback (unknown toc or complete resolution failure):
        Return empty set → is_extraterritorial defaults to False (conservative).
    """
    toc = row.get("type_of_conflict", None)

    if toc in (3, 4):
        # Intrastate / internationalised: location = side_a's territory
        side_a_str = row.get("side_a", "")
        iso3_list, is_state = _resolve_ucdp_actor(side_a_str, res, dataset="ucdp_dyadic")
        resolved = {iso3 for iso3 in iso3_list if iso3 is not None}
        if not resolved:
            # side_a failed resolution — fall back to gwno_loc
            resolved = gw_to_iso3_set(row.get("gwno_loc", None))
        return resolved

    elif toc == 2:
        # Interstate — detect coalition invasion pattern
        side_a_str = row.get("side_a", "")
        side_a_iso3s = [iso3 for iso3 in
                        _resolve_ucdp_actor(side_a_str, res, dataset="ucdp_dyadic")[0]
                        if iso3 is not None]
        if len(side_a_iso3s) > 1:
            # Multi-state coalition on side_a: UCDP encodes all coalition home GW codes
            # in gwno_loc, making coalition members appear non-extraterritorial.
            # Use side_b's territory (the invaded / defending country) instead.
            side_b_str = row.get("side_b", "")
            side_b_resolved = {iso3 for iso3 in
                               _resolve_ucdp_actor(side_b_str, res, dataset="ucdp_dyadic")[0]
                               if iso3 is not None}
            return side_b_resolved if side_b_resolved else gw_to_iso3_set(row.get("gwno_loc", None))
        else:
            # Bilateral war: gwno_loc contains both parties' territories (correct)
            return gw_to_iso3_set(row.get("gwno_loc", None))

    elif toc == 1:
        # Extrasystemic: gwno_loc is reliable
        return gw_to_iso3_set(row.get("gwno_loc", None))

    else:
        # Unknown type — conservative fallback
        return set()


# ── Unit tests — _conflict_location_iso3_set routing ─────────────────────────
_test_rows = [
    # Iraq War 2003: type 2, multi-state side_a (AUS+GBR+USA coalition), gwno_loc encodes all
    {
        "type_of_conflict": 2,
        "side_a": "Government of Australia, Government of United Kingdom, Government of United States of America",
        "side_b": "Government of Iraq",
        "gwno_loc": "2, 200, 645, 900",
        "expected_location": {"IRQ"},
        "label": "Iraq 2003 (type 2, multi-state coalition) → should resolve to {IRQ} only",
    },
    # Syria civil war with Russian intervention: type 4, side_a = Government of Syria
    {
        "type_of_conflict": 4,
        "side_a": "Government of Syria",
        "side_b": "Syrian insurgents",
        "gwno_loc": "365, 652",
        "expected_location": {"SYR"},
        "label": "Syria 2015 (type 4) → should resolve to {SYR} only",
    },
    # Kargil 1999: type 2 bilateral (single-state side_a), gwno_loc has both countries
    {
        "type_of_conflict": 2,
        "side_a": "Government of India",
        "side_b": "Government of Pakistan",
        "gwno_loc": "750, 770",
        "expected_location": {"IND", "PAK"},
        "label": "Kargil 1999 (type 2, bilateral) → should resolve to {IND, PAK}",
    },
    # Pure intrastate: type 3, side_a = Government of Colombia
    {
        "type_of_conflict": 3,
        "side_a": "Government of Colombia",
        "side_b": "FARC",
        "gwno_loc": "100",
        "expected_location": {"COL"},
        "label": "Colombia type 3 → should resolve to {COL}",
    },
]

_r2 = ISO3Resolver()
print("\n_conflict_location_iso3_set routing tests:")
_all_routing_pass = True
for _t in _test_rows:
    _row = pd.Series(_t)
    _result = _conflict_location_iso3_set(_row, _r2)
    _passed = _result == _t["expected_location"]
    if not _passed:
        _all_routing_pass = False
    _status = "PASS" if _passed else "FAIL"
    print(f"  [{_status}] {_t['label']}")
    if not _passed:
        print(f"         Got: {_result}, Expected: {_t['expected_location']}")

assert _all_routing_pass, "Routing unit tests FAILED — fix _conflict_location_iso3_set"
print("Routing unit tests complete.")

All _resolve_ucdp_actor unit tests passed.

_conflict_location_iso3_set routing tests:
  [PASS] Iraq 2003 (type 2, multi-state coalition) → should resolve to {IRQ} only
  [PASS] Syria 2015 (type 4) → should resolve to {SYR} only
  [PASS] Kargil 1999 (type 2, bilateral) → should resolve to {IND, PAK}
  [PASS] Colombia type 3 → should resolve to {COL}
Routing unit tests complete.


In [9]:
def _clean_dyadic(df, res):
    df = df.copy()
    df["year"] = df["year"].astype(int)

    # Resolve conflict location (primary country where fighting occurs)
    df["iso3"] = res.resolve_series(df["location"], dataset="ucdp_dyadic")

    # Resolve side actors to ISO3 lists (state actors only; non-state → None)
    def _side_iso3s(actor_str):
        iso3s, _ = _resolve_ucdp_actor(actor_str, res, dataset="ucdp_dyadic")
        return [x for x in iso3s if x is not None]

    df["side_a_iso3"] = df["side_a"].apply(_side_iso3s)
    df["side_b_iso3"] = df["side_b"].apply(_side_iso3s)
    df["side_a_2nd_iso3"] = df["side_a_2nd"].fillna("").apply(_side_iso3s)
    df["side_b_2nd_iso3"] = df["side_b_2nd"].fillna("").apply(_side_iso3s)
    return df

ucdp_dyadic_clean = _clean_dyadic(ucdp_dyadic, resolver)
print(f"UCDP Dyadic: {len(ucdp_dyadic_clean):,} rows")
print(f"  iso3 (location) resolved: {ucdp_dyadic_clean['iso3'].notna().sum():,}")
save_checkpoint(ucdp_dyadic_clean, CLEAN_DIR / "ucdp_dyadic_clean.parquet")


UCDP Dyadic: 3,432 rows
  iso3 (location) resolved: 3,279
[checkpoint] saved → ucdp_dyadic_clean.parquet  (3,432 rows)


### 3.5 SIPRI Military Expenditure (MILEX)

SIPRI uses non-standard country names in several cases: `"Korea, South"`, `"Congo, DR"`,
`"Viet Nam"`, `"European Union"` (supra-national aggregate → drop), and the UTF-8
variant `"Türkiye"`. These are handled via SIPRI-specific `dataset_overrides` passed
to a dedicated resolver instance, supplementing `GLOBAL_OVERRIDES`.


In [10]:
SIPRI_OVERRIDES = {
    "Korea, South":          "KOR",
    "Kong, DR":              "COD",
    "Congo, DR":             "COD",
    "Congo, Republic":       "COG",
    "Viet Nam":              "VNM",
    "Timor Leste":           "TLS",
    "Kyrgyz Republic":       "KGZ",
    "Gambia, The":           "GMB",
    "Cote d'Ivoire":         "CIV",
    "European Union":        None,
    "German Democratic Republic": "DEU",
    "USSR":                  "RUS",
}
milex_resolver = ISO3Resolver(dataset_overrides=SIPRI_OVERRIDES)

milex = sipri_milex.copy()
milex["iso3"] = milex_resolver.resolve_series(milex["Country"], dataset="sipri_milex")
milex["year"] = milex["year"].astype(int)
milex = milex.rename(columns={"Country": "country_name"})

n_dropped = milex["iso3"].isna().sum()
sipri_milex_clean = milex.dropna(subset=["iso3"]).reset_index(drop=True)
print(f"SIPRI MILEX: {len(milex):,} rows  |  dropped (unresolved / aggregates): {n_dropped:,}")
print(f"  iso3 resolved: {len(sipri_milex_clean):,}")

# Fix 2: Deduplicate (iso3, year) from historical-state merges (e.g. GFR+GDR→DEU).
# milex_const2024_usd is additive — the correct value for unified Germany is the
# sum of West + East Germany expenditures, not the average.
_dupes = sipri_milex_clean.duplicated(subset=["iso3", "year"]).sum()
if _dupes > 0:
    print(f"  Fix 2: deduplicating {_dupes} historical-state (iso3,year) rows — sum for milex")
    sipri_milex_clean = (
        sipri_milex_clean
        .groupby(["iso3", "year"], as_index=False)
        .agg({"milex_const2024_usd": "sum", "country_name": "first"})
        .reset_index(drop=True)
    )
else:
    print("  No (iso3, year) duplicates in SIPRI MILEX.")
save_checkpoint(sipri_milex_clean, CLEAN_DIR / "sipri_milex_clean.parquet")


SIPRI MILEX: 8,435 rows  |  dropped (unresolved / aggregates): 43
  iso3 resolved: 8,392
  Fix 2: deduplicating 30 historical-state (iso3,year) rows — sum for milex
[checkpoint] saved → sipri_milex_clean.parquet  (8,362 rows)


### 3.6 SIPRI TIV Transfer Register

Both `supplier` and `recipient` columns are resolved to ISO3.
Non-state actors in SIPRI are identified by:
  - An asterisk (`*`) at the end of the name (SIPRI convention for non-state groups)
  - Failed ISO3 resolution

Non-state recipient rows are split off into `sipri_tiv_non_state.parquet` so
downstream analysis operates only on state-to-state transfers.
Supplier ISO3 → `supplier_iso3`; Recipient ISO3 → `recipient_iso3`.


In [11]:
reg = sipri_tiv_reg.copy()

reg["supplier_iso3"] = milex_resolver.resolve_series(reg["supplier"], dataset="sipri_tiv")
reg["recipient_iso3"] = milex_resolver.resolve_series(reg["recipient"], dataset="sipri_tiv")

# Non-state flag: asterisk in name OR failed resolution
reg["_has_asterisk"] = reg["recipient"].str.contains(r"\*", na=False)
reg["_non_state"] = reg["_has_asterisk"] | reg["recipient_iso3"].isna()

sipri_tiv_non_state     = reg[reg["_non_state"]].drop(columns=["_has_asterisk", "_non_state"])
sipri_tiv_register_clean = reg[~reg["_non_state"]].drop(columns=["_has_asterisk", "_non_state"])

sipri_tiv_register_clean["year"] = sipri_tiv_register_clean["year"].astype(int)
sipri_tiv_non_state["year"]      = sipri_tiv_non_state["year"].astype(int)

print(f"SIPRI TIV register: {len(reg):,} total")
print(f"  state-to-state rows: {len(sipri_tiv_register_clean):,}")
print(f"  non-state recipient rows: {len(sipri_tiv_non_state):,}")
print(f"  non-state recipients: {sorted(sipri_tiv_non_state['recipient'].unique())[:10]}")

save_checkpoint(sipri_tiv_register_clean, CLEAN_DIR / "sipri_tiv_register_clean.parquet")
save_checkpoint(sipri_tiv_non_state,      CLEAN_DIR / "sipri_tiv_non_state.parquet")


SIPRI TIV register: 60,789 total
  state-to-state rows: 60,165
  non-state recipient rows: 624
  non-state recipients: ['ANC (South Africa)*', 'African Union**', 'Amal (Lebanon)*', 'Anti-Castro rebels (Cuba)*', 'Armas (Guatemala)*', 'Biafra', 'Contras (Nicaragua)*', 'Darfur rebels (Sudan)*', 'ELF (Ethiopia)*', 'EPLF (Ethiopia)*']
[checkpoint] saved → sipri_tiv_register_clean.parquet  (60,165 rows)
[checkpoint] saved → sipri_tiv_non_state.parquet  (624 rows)


### 3.7 SIPRI TIV Aggregations

Re-derived from `sipri_tiv_register_clean` (non-state rows already excluded)
so the filtering propagates into both aggregated outputs.


In [12]:
# Aggregation 1: recipient × year → total TIV imports
tiv_by_cy = (
    sipri_tiv_register_clean
    .groupby(["recipient_iso3", "year"], as_index=False)["tiv"]
    .sum()
    .rename(columns={"recipient_iso3": "iso3", "tiv": "tiv_imports_total"})
    .sort_values(["iso3", "year"])
    .reset_index(drop=True)
)
tiv_by_cy["year"] = tiv_by_cy["year"].astype(int)

# Aggregation 2: recipient × year × weapon_category pivot
by_cwc = sipri_tiv_register_clean.groupby(
    ["recipient_iso3", "year", "weapon_category"], as_index=False
)["tiv"].sum()

tiv_by_cyc = by_cwc.pivot_table(
    index=["recipient_iso3", "year"],
    columns="weapon_category",
    values="tiv",
    aggfunc="sum",
    fill_value=0,
).reset_index()
tiv_by_cyc.columns.name = None
tiv_by_cyc.columns = [
    c if c in ("recipient_iso3", "year")
    else c.lower().replace(" ", "_").replace("-", "_")
    for c in tiv_by_cyc.columns
]
tiv_by_cyc = tiv_by_cyc.rename(columns={"recipient_iso3": "iso3"})
tiv_by_cyc["year"] = tiv_by_cyc["year"].astype(int)

print(f"TIV by country-year:          {tiv_by_cy.shape}")
print(f"TIV by country-year-category: {tiv_by_cyc.shape}")

save_checkpoint(tiv_by_cy,  CLEAN_DIR / "sipri_tiv_by_country_year_clean.parquet")
save_checkpoint(tiv_by_cyc, CLEAN_DIR / "sipri_tiv_by_country_year_category_clean.parquet")


TIV by country-year:          (7845, 3)
TIV by country-year-category: (7845, 13)
[checkpoint] saved → sipri_tiv_by_country_year_clean.parquet  (7,845 rows)
[checkpoint] saved → sipri_tiv_by_country_year_category_clean.parquet  (7,845 rows)


### 3.8 World Bank WDI

The WDI download includes 265 entries: ~217 country-level and ~48 aggregate/regional
entities ("World", "High income", "East Asia & Pacific", etc.).
Aggregates resolve to `None` via `ISO3Resolver` and are dropped.


In [13]:
wdi = worldbank_wdi.copy()
wdi["iso3"] = resolver.resolve_series(wdi["country"], dataset="worldbank_wdi")
wdi["year"] = wdi["year"].astype(int)

n_total   = len(wdi)
n_no_iso3 = wdi["iso3"].isna().sum()
worldbank_wdi_clean = wdi.dropna(subset=["iso3"]).reset_index(drop=True)

print(f"WDI: {n_total:,} rows total")
print(f"  dropped (aggregates / unresolved): {n_no_iso3:,}")
print(f"  kept (country-level):              {len(worldbank_wdi_clean):,}")
print(f"  unique countries after clean:      {worldbank_wdi_clean['iso3'].nunique():,}")

# Spot-check dropped names (should all be regional/income aggregates)
dropped_names = wdi[wdi["iso3"].isna()]["country"].unique()
print(f"\n  Sample dropped names ({len(dropped_names)} unique):")
for n in sorted(dropped_names)[:10]:
    print(f"    {n!r}")


# Fix 2: Deduplicate (iso3, year) from historical-state merges.
# Policy: gdp_usd and population are additive (sum); gdp_per_capita is a ratio (mean).
_dupes = worldbank_wdi_clean.duplicated(subset=["iso3", "year"]).sum()
if _dupes > 0:
    print(f"  Fix 2: deduplicating {_dupes} historical-state (iso3,year) rows in WDI")
    worldbank_wdi_clean = (
        worldbank_wdi_clean
        .groupby(["iso3", "year"], as_index=False)
        .agg({"gdp_usd": "sum", "gdp_per_capita": "mean",
              "population": "sum", "country": "first"})
        .reset_index(drop=True)
    )
else:
    print("  No (iso3, year) duplicates in WDI.")
save_checkpoint(worldbank_wdi_clean, CLEAN_DIR / "worldbank_wdi_clean.parquet")


WDI: 17,195 rows total
  dropped (aggregates / unresolved): 4,160
  kept (country-level):              13,035
  unique countries after clean:      201

  Sample dropped names (64 unique):
    'Africa Eastern and Southern'
    'Africa Western and Central'
    'Arab World'
    'Caribbean small states'
    'Central Europe and the Baltics'
    'Channel Islands'
    'Curacao'
    'Early-demographic dividend'
    'East Asia & Pacific'
    'East Asia & Pacific (IDA & IBRD countries)'
  No (iso3, year) duplicates in WDI.
[checkpoint] saved → worldbank_wdi_clean.parquet  (13,035 rows)


### 3.9 V-Dem Core Dataset

`country_text_id` is V-Dem's 3-letter code. For current states it matches ISO 3166-1
alpha-3, but V-Dem uses its own codes for historical entities (e.g. `DDR` for East
Germany, `SER` for pre-2006 Serbia). We:

1. Verify each unique code against pycountry — codes that fail are printed for review.
2. Apply `GLOBAL_OVERRIDES` / a V-Dem-specific override dict for historical states.
3. Handle duplicate country-year rows from state transitions: if the same `iso3` and
   `year` appear twice (e.g. Yemen unification 1990, Czechoslovakia split 1993), keep
   the row with higher `v2x_polyarchy` data coverage (non-NaN preferred; else max value).


In [14]:
VDEM_OVERRIDES = {
    "DDR": "DEU",   # East Germany
    "GDR": "DEU",   # East Germany (alternate V-Dem code)
    "CZS": "CZE",   # Czechoslovakia
    "YUG": "SRB",   # Yugoslavia
    "SSU": "SSD",   # South Sudan (if present pre-2011)
    "SML": None,    # Somaliland
    "ZZB": "SRB",   # Yugoslavia (alternate)
    "PSG": "PSE",   # Palestine
}
vdem_resolver = ISO3Resolver(dataset_overrides=VDEM_OVERRIDES)

df_vdem = vdem.copy()

# Report any unmatched V-Dem codes
unique_codes = sorted(df_vdem["country_text_id"].unique())
unmatched_vdem = []
for code in unique_codes:
    iso3 = vdem_resolver.resolve(code, dataset="vdem")
    if iso3 is None:
        unmatched_vdem.append(code)

if unmatched_vdem:
    print(f"V-Dem codes with no ISO3 match ({len(unmatched_vdem)}): {unmatched_vdem}")
    print("Add these to VDEM_OVERRIDES or GLOBAL_OVERRIDES as needed.")
else:
    print("All V-Dem country_text_id codes resolved successfully.")

df_vdem["iso3"] = vdem_resolver.resolve_series(df_vdem["country_text_id"], dataset="vdem")
df_vdem["year"] = df_vdem["year"].astype(int)

# Handle duplicate (iso3, year) from state transitions
# Policy: keep the row with the higher v2x_polyarchy coverage (non-NaN preferred, then max)
before = len(df_vdem)
df_vdem = df_vdem.sort_values(
    ["iso3", "year", "v2x_polyarchy"],
    ascending=[True, True, False],
    na_position="last",
)
df_vdem = df_vdem.drop_duplicates(subset=["iso3", "year"], keep="first")
after = len(df_vdem)
print(f"\nV-Dem: {before:,} → {after:,} rows after deduplication ({before-after} duplicates removed)")

# Drop rows where iso3 could not be resolved (historical entities with no clean successor)
vdem_clean = df_vdem.dropna(subset=["iso3"]).reset_index(drop=True)
print(f"  Dropped {after - len(vdem_clean)} rows with unresolvable iso3")

save_checkpoint(vdem_clean, CLEAN_DIR / "vdem_clean.parquet")


V-Dem codes with no ISO3 match (5): ['PSB', 'SML', 'VDR', 'XKX', 'YMD']
Add these to VDEM_OVERRIDES or GLOBAL_OVERRIDES as needed.

V-Dem: 13,080 → 12,866 rows after deduplication (214 duplicates removed)
  Dropped 79 rows with unresolvable iso3
[checkpoint] saved → vdem_clean.parquet  (12,787 rows)


### 3.10 CoW CINC

The `cow` DataFrame was already cleaned and ISO3-mapped in Section 2.
Drop rows with unresolved `iso3` (historical micro-states that map to `None`)
and save.


In [15]:
cow_cinc_clean = cow.dropna(subset=["iso3"]).copy()
cow_cinc_clean["year"] = cow_cinc_clean["year"].astype(int)
print(f"COW CINC clean: {len(cow_cinc_clean):,} rows  |  unique iso3: {cow_cinc_clean['iso3'].nunique()}")

# Fix 2: Deduplicate (iso3, year) from historical-state merges (e.g. GFR+GDR→DEU).
# milper, milex, cinc are additive quantities; stateabb → first.
# Replace COW missing-data sentinel -9 with NaN before summing.
for _col in ["milper", "milex", "cinc"]:
    if _col in cow_cinc_clean.columns:
        cow_cinc_clean[_col] = cow_cinc_clean[_col].replace(-9, float("nan"))
_dupes = cow_cinc_clean.duplicated(subset=["iso3", "year"]).sum()
if _dupes > 0:
    print(f"  Fix 2: deduplicating {_dupes} historical-state (iso3,year) rows in CoW CINC")
    cow_cinc_clean = (
        cow_cinc_clean
        .groupby(["iso3", "year"], as_index=False)
        .agg({"milper": "sum", "milex": "sum", "cinc": "sum", "stateabb": "first"})
        .reset_index(drop=True)
    )
else:
    print("  No (iso3, year) duplicates in CoW CINC.")
save_checkpoint(cow_cinc_clean, CLEAN_DIR / "cow_cinc_clean.parquet")


COW CINC clean: 15,393 rows  |  unique iso3: 190
  Fix 2: deduplicating 227 historical-state (iso3,year) rows in CoW CINC
[checkpoint] saved → cow_cinc_clean.parquet  (15,166 rows)


---
## Section 4 — UCDP Participation Panel

### Why this panel matters

The raw dyadic dataset records *dyads* (A vs B pairs), not individual country
participation. A country that sends troops abroad appears as a `side_a_2nd` or
`side_b_2nd` supporter — it would be missed if we only looked at the primary sides.

This section builds a **long-form participation table** where each row is one
`(iso3, year, conflict)` triple. The key analytical variable is:

> **`is_extraterritorial`** = `True` when the participant's `iso3` does not match
> the conflict's location `iso3`. This cleanly separates countries fighting *on
> their own soil* from those projecting force abroad — the structural distinction
> at the heart of the Stability-Instability Paradox.
>
> Five validated spot-checks (v3):
> - USA in Iraq 2003 (conflict 420, type 4) → `is_extraterritorial = True`
> - GBR in Iraq 2003 (conflict 420, type 4) → `is_extraterritorial = True`
> - AUS in Iraq 2003 (conflict 420, type 4) → `is_extraterritorial = True`
> - Russia in Syria 2015 → `is_extraterritorial = True`
> - India in Kargil 1999 → `is_extraterritorial = False`

**Location encoding method (v3 fix — ToC routing):** `is_extraterritorial` is
resolved via `_conflict_location_iso3_set()`, which routes by `type_of_conflict`:
- **Types 3 & 4** (intrastate / internationalised): location = `side_a`'s territory.
  `side_a` is the defending government by UCDP convention, so resolving it to ISO3
  yields the territorial host cleanly, bypassing the `gwno_loc` coalition-encoding
  problem (where UCDP folds intervening states' GW codes into `gwno_loc` for
  internationalised conflicts).
- **Types 1 & 2** (extrasystemic / interstate): `gwno_loc` is reliable and is
  used directly via `gw_to_iso3_set()`.

This correctly marks all major coalition expeditionary operations (Iraq 2003,
Afghanistan 2001–, Libya 2011) as extraterritorial for all participating states,
raising the extraterritorial count relative to any single-field approach.
Reported in the limitations section with the note that interstate conflicts where
both states fight on contested territory remain an acknowledged edge case.

In [16]:
def _build_participation_panel(dyadic_df, res):
    """Build long-form (iso3, year, conflict_id) participation table.

    is_extraterritorial = True when the participant's iso3 is not in the
    set of ISO3 codes returned by _conflict_location_iso3_set() for that row.

    v3 fix (ToC routing): location resolution is now routed by type_of_conflict.
    Types 3 & 4 use side_a's territory (the defending government's country).
    Types 1 & 2 use gwno_loc directly (reliable for extrasystemic/interstate).
    This correctly marks all coalition participants (USA, GBR, AUS in Iraq 2003)
    as extraterritorial while preserving correct non-extraterritorial flags for
    interstate conflicts where both sides fight on their own soil (Kargil 1999).
    """
    records = []

    for _, row in dyadic_df.iterrows():
        year         = int(row["year"])
        conflict_id  = row["conflict_id"]
        dyad_id      = row["dyad_id"]
        toc          = row["type_of_conflict"]
        intensity    = row["intensity_level"]

        # Resolve conflict location using ToC-aware routing.
        # _conflict_location_iso3_set() routes by type_of_conflict:
        #   type 3/4 → side_a's territory (avoids gwno_loc coalition-encoding problem)
        #   type 1/2 → gwno_loc directly (reliable for interstate/extrasystemic)
        location_iso3_set = _conflict_location_iso3_set(row, res)

        # Emit one record per state participant on each side
        side_cols = [
            ("side_a",     "A"),
            ("side_b",     "B"),
            ("side_a_2nd", "A"),
            ("side_b_2nd", "B"),
        ]
        for col, role in side_cols:
            actor_str = row.get(col, "")
            if not isinstance(actor_str, str) or not actor_str.strip():
                continue
            iso3_list, _is_state = _resolve_ucdp_actor(actor_str, res, dataset="ucdp_dyadic")
            for iso3 in iso3_list:
                if iso3 is None:
                    continue
                # is_extraterritorial: participant fighting outside their own territory.
                # Defaults to False when location_iso3_set is empty (conservative).
                is_extra = bool(location_iso3_set) and (iso3 not in location_iso3_set)
                records.append({
                    "iso3":              iso3,
                    "year":              year,
                    "conflict_id":       conflict_id,
                    "dyad_id":           dyad_id,
                    "role":              role,
                    "type_of_conflict":  toc,
                    "intensity_level":   intensity,
                    "is_extraterritorial": is_extra,
                })

    panel = pd.DataFrame(records)
    if panel.empty:
        return panel
    panel["year"] = panel["year"].astype(int)
    # Deduplicate: same (iso3, year, conflict, dyad, role) from 2nd-party processing
    return panel.drop_duplicates(
        subset=["iso3", "year", "conflict_id", "dyad_id", "role"]
    ).reset_index(drop=True)


print("Building participation panel — this iterates over 3,432 dyadic rows …")
participation_long = _build_participation_panel(ucdp_dyadic_clean, resolver)
print(f"participation_long: {len(participation_long):,} rows  |  "
      f"unique iso3: {participation_long['iso3'].nunique()}  |  "
      f"extraterritorial: {participation_long['is_extraterritorial'].sum():,}")

Building participation panel — this iterates over 3,432 dyadic rows …
participation_long: 8,157 rows  |  unique iso3: 160  |  extraterritorial: 4,720


In [17]:
# ── Spot-check: three known extraterritorial cases ────────────────────────────
print("=== Spot-check: is_extraterritorial cases ===\n")

# 1. USA in Iraq 2003 — USA is fighting in Iraq (IRQ), not on US soil
usa_iraq = participation_long[
    (participation_long["iso3"] == "USA") &
    (participation_long["year"] == 2003)
]
print("USA in 2003 conflicts:")
print(usa_iraq[["iso3", "year", "conflict_id", "role", "is_extraterritorial"]].to_string(index=False))

# 2. India in 1999 — Kargil war (location=India); India is on its own soil
india_99 = participation_long[
    (participation_long["iso3"] == "IND") &
    (participation_long["year"] == 1999) &
    (participation_long["type_of_conflict"] == 2)   # interstate
]
print("\nIndia in 1999 interstate conflicts (Kargil):")
print(india_99[["iso3", "year", "conflict_id", "role", "is_extraterritorial"]].to_string(index=False))

# 3. Russia in Syria 2015 — Russia joined pro-government side; location=Syria
rus_2015 = participation_long[
    (participation_long["iso3"] == "RUS") &
    (participation_long["year"] == 2015)
]
print("\nRussia in 2015 conflicts:")
print(rus_2015[["iso3", "year", "conflict_id", "role", "is_extraterritorial"]].to_string(index=False))


=== Spot-check: is_extraterritorial cases ===

USA in 2003 conflicts:
iso3  year  conflict_id role  is_extraterritorial
 USA  2003          333    A                 True
 USA  2003          333    A                 True
 USA  2003          418    A                False
 USA  2003          420    A                 True

India in 1999 interstate conflicts (Kargil):
iso3  year  conflict_id role  is_extraterritorial
 IND  1999          218    A                False

Russia in 2015 conflicts:
iso3  year  conflict_id role  is_extraterritorial
 RUS  2015          299    A                 True
 RUS  2015        13306    B                 True
 RUS  2015        13588    A                False
 RUS  2015        13604    A                 True
 RUS  2015        13306    B                 True
 RUS  2015        13306    B                 True
 RUS  2015          432    A                False


### Fix 1 — Russia in Syria 2015: Case A diagnosis and resolution

**Diagnostic (2026-05-16):** The spot-check above initially returned an empty DataFrame
for Russia in 2015. Inspection of `ucdp_dyadic_raw.parquet` revealed:

```
conflict_id 299 / 13604 — side_a_2nd:
  "Government of Iran, Government of Russia (Soviet Union)"
```

Russia *does* appear in UCDP Dyadic v25.1 for Syria 2015 (and also 2016), but
under the parenthetical form `"Government of Russia (Soviet Union)"`.

**Root cause:** `_resolve_ucdp_actor()` correctly strips `"Government of "`, leaving
`"Russia (Soviet Union)"`. This parenthetical-suffixed form had no entry in
`GLOBAL_OVERRIDES` and failed all three pycountry exact-match lookups, so it
silently resolved to `None` — dropping Russia from the participation panel entirely.

**Fix applied (Fix 2 / `src/iso3.py`):** A `re.sub(r"\s*\([^)]*\)\s*$", "", ...)` step
was added at the top of `ISO3Resolver.resolve()`, **before all three lookup tiers**.
This converts `"Russia (Soviet Union)"` → `"Russia"`, which matches
`GLOBAL_OVERRIDES["Russia"] = "RUS"`.

The same strip fixes 30+ other parenthetical variants in UCDP:
`"Cambodia (Kampuchea)"`, `"Serbia (Yugoslavia)"`, `"Vietnam (North Vietnam)"`, etc.

**After fix:** Russia appears in Syria 2015 with `is_extraterritorial = True` (correct:
Russia fought in SYR, not on its own territory). This is a **pipeline bug**, not a
source-data limitation — UCDP v25.1 codes Russia's Syria intervention from 2015 onward.

In [18]:
# ── Audit: ToC-routing fix validation ─────────────────────────────────────────
# Five spot-checks covering all routing branches:
#   conflict 420, type 2 multi-state coalition  → USA, GBR, AUS extraterritorial
#   conflict 299, type 4 unilateral             → RUS extraterritorial
#   conflict 218, type 2 bilateral              → IND NOT extraterritorial
# Note: RUS is pinned to conflict 299 (Syria) because Russia also has domestic
# conflicts in 2015 (conflict 432 = Caucasus Emirate, in Russia) where
# is_extraterritorial=False is correct — Russia fighting at home is not extraterritorial.

print("=== ToC-routing fix validation ===\n")

cases = [
    ("USA", 2003, 420, True,  "USA in Iraq 2003 (type 2, multi-state coalition) → extraterritorial"),
    ("GBR", 2003, 420, True,  "UK in Iraq 2003 (type 2, multi-state coalition) → extraterritorial"),
    ("AUS", 2003, 420, True,  "Australia in Iraq 2003 (type 2, multi-state coalition) → extraterritorial"),
    ("RUS", 2015, 299, True,  "Russia in Syria 2015 (conflict 299, type 4 unilateral) → extraterritorial"),
    ("IND", 1999, 218, False, "India in Kargil 1999 (type 2 bilateral) → NOT extraterritorial"),
]

all_passed = True
for iso3, year, conflict_id, expected_extra, label in cases:
    mask = (
        (participation_long["iso3"] == iso3) &
        (participation_long["year"] == year) &
        (participation_long["conflict_id"] == conflict_id)
    )
    rows = participation_long[mask]

    if rows.empty:
        print(f"[WARN — NO ROWS] {label}")
        all_passed = False
        continue

    actual_vals = rows["is_extraterritorial"].unique()
    consistent = len(actual_vals) == 1
    matches_expected = bool(actual_vals[0]) == expected_extra if consistent else False
    status = "PASS" if (consistent and matches_expected) else "FAIL"
    if not (consistent and matches_expected):
        all_passed = False

    print(f"[{status}] {label}")
    print(rows[["iso3","year","conflict_id","type_of_conflict","role","is_extraterritorial"]].to_string())
    print()

print("─" * 60)
print(f"All spot-checks passed: {all_passed}")
print()

# Quantitative summary
n_extra = participation_long["is_extraterritorial"].sum()
print(f"Total extraterritorial participations: {n_extra:,}")
print(f"Baseline (original location-text method):  4,270")
print(f"After gwno_loc fix (previous session):     4,914")
print(f"After ToC-routing fix (this session):      {n_extra:,}")
print(f"Net delta from original: {n_extra - 4270:+,}")
print("Positive delta expected — multilateral coalition ops now counted correctly.")

# Distribution by type_of_conflict
print("\nExtraterritorial count by type_of_conflict:")
print(
    participation_long[participation_long["is_extraterritorial"]]
    .groupby("type_of_conflict")
    .size()
    .rename_axis("toc")
    .rename("n_extraterritorial")
    .to_string()
)
print("(Type 4 should dominate — most extraterritorial action is internationalised intrastate)")

=== ToC-routing fix validation ===

[PASS] USA in Iraq 2003 (type 2, multi-state coalition) → extraterritorial
     iso3  year  conflict_id  type_of_conflict role  is_extraterritorial
8040  USA  2003          420                 2    A                 True

[PASS] UK in Iraq 2003 (type 2, multi-state coalition) → extraterritorial
     iso3  year  conflict_id  type_of_conflict role  is_extraterritorial
8039  GBR  2003          420                 2    A                 True

[PASS] Australia in Iraq 2003 (type 2, multi-state coalition) → extraterritorial
     iso3  year  conflict_id  type_of_conflict role  is_extraterritorial
8038  AUS  2003          420                 2    A                 True

[PASS] Russia in Syria 2015 (conflict 299, type 4 unilateral) → extraterritorial
    iso3  year  conflict_id  type_of_conflict role  is_extraterritorial
128  RUS  2015          299                 4    A                 True

[PASS] India in Kargil 1999 (type 2 bilateral) → NOT extraterritori

In [19]:
# ── Aggregate to country-year ──────────────────────────────────────────────────
participation_by_country_year = (
    participation_long
    .groupby(["iso3", "year"])
    .agg(
        n_conflicts        =("conflict_id",         "nunique"),
        n_interstate       =("type_of_conflict",    lambda x: (x == 2).sum()),
        n_intrastate       =("type_of_conflict",    lambda x: (x == 3).sum()),
        n_internationalised=("type_of_conflict",    lambda x: (x == 4).sum()),
        n_extrasystemic    =("type_of_conflict",    lambda x: (x == 1).sum()),
        n_war_intensity    =("intensity_level",     lambda x: (x == 2).sum()),
        n_minor_intensity  =("intensity_level",     lambda x: (x == 1).sum()),
        n_extraterritorial =("is_extraterritorial", "sum"),
    )
    .reset_index()
)
participation_by_country_year["year"] = participation_by_country_year["year"].astype(int)

print(f"participation_by_country_year: {len(participation_by_country_year):,} rows  |  "
      f"iso3: {participation_by_country_year['iso3'].nunique()}  |  "
      f"years: {int(participation_by_country_year['year'].min())}–"
      f"{int(participation_by_country_year['year'].max())}")

save_checkpoint(participation_long,               CLEAN_DIR / "ucdp_participation_long.parquet")
save_checkpoint(participation_by_country_year,    CLEAN_DIR / "ucdp_participation_by_country_year.parquet")


participation_by_country_year: 3,505 rows  |  iso3: 160  |  years: 1946–2024
[checkpoint] saved → ucdp_participation_long.parquet  (8,157 rows)
[checkpoint] saved → ucdp_participation_by_country_year.parquet  (3,505 rows)


---
## Section 5 — Per-Source QA Printouts

Standardised diagnostic block for every clean file.


In [20]:
def _qa(label, df_raw, df_clean, iso3_col="iso3", year_col="year"):
    n_raw    = len(df_raw)
    n_clean  = len(df_clean)
    n_iso3   = df_clean[iso3_col].notna().sum() if iso3_col in df_clean.columns else n_clean
    pct      = 100 * n_iso3 / n_clean if n_clean else 0
    n_drop   = n_raw - n_clean
    yr_min   = int(df_clean[year_col].min()) if year_col in df_clean.columns else "?"
    yr_max   = int(df_clean[year_col].max()) if year_col in df_clean.columns else "?"
    top10    = df_clean[iso3_col].value_counts().head(10).to_dict() if iso3_col in df_clean.columns else {}
    unmatched = [
        f"{r['name']} ({r['count']})"
        for _, r in resolver.report_unmatched().iterrows()
        if r["dataset"] == label
    ]
    print(f"[{label}]  {n_raw:,} rows → {n_iso3:,} with iso3 ({pct:.1f}%), {n_drop:,} dropped")
    print(f"  year range: {yr_min}–{yr_max}")
    if top10:
        print(f"  top 10 iso3: {top10}")
    if unmatched:
        print(f"  unmatched names: {unmatched[:10]}")

_qa("ucdp_acd",    ucdp_acd,      ucdp_acd_clean)
_qa("ucdp_ged",    ucdp_ged,      ucdp_ged_clean)
_qa("ucdp_brd",    ucdp_brd,      ucdp_brd_clean)
_qa("ucdp_dyadic", ucdp_dyadic,   ucdp_dyadic_clean)
_qa("sipri_milex", sipri_milex,   sipri_milex_clean,   iso3_col="iso3")
_qa("sipri_tiv",   sipri_tiv_reg, sipri_tiv_register_clean, iso3_col="supplier_iso3")
_qa("worldbank_wdi", worldbank_wdi, worldbank_wdi_clean)
_qa("vdem",        vdem,          vdem_clean)
_qa("cow_cinc",    cow_cinc,      cow_cinc_clean)


[ucdp_acd]  2,752 rows → 2,914 with iso3 (99.9%), -165 dropped
  year range: 1946–2024
  top 10 iso3: {'MMR': 304, 'IND': 218, 'ETH': 140, 'PHL': 121, 'ISR': 100, 'IRQ': 81, 'PAK': 79, 'IRN': 66, 'AFG': 60, 'COL': 60}
[ucdp_ged]  385,918 rows → 385,918 with iso3 (100.0%), 0 dropped
  year range: 1989–2024
  top 10 iso3: {'SYR': 87861, 'AFG': 42220, 'UKR': 31547, 'MEX': 21550, 'IND': 17997, 'COL': 14620, 'IRQ': 9438, 'BIH': 9340, 'COD': 8901, 'MMR': 8476}
[ucdp_brd]  1,586 rows → 1,536 with iso3 (96.8%), 0 dropped
  year range: 1989–2024
  top 10 iso3: {'IND': 142, 'MMR': 109, 'PHL': 73, 'ETH': 66, 'PAK': 46, 'AFG': 46, 'TUR': 44, 'ISR': 42, 'COD': 38, 'SDN': 36}
  unmatched names: ['Afghanistan, Pakistan (1)', 'Afghanistan, United Kingdom, United States of America (1)', 'Australia, Iraq, United Kingdom, United States of America (1)', 'Cambodia (Kampuchea), Thailand (1)', 'Cameroon, Nigeria (1)', 'China, India (1)', 'Djibouti, Eritrea (1)', 'Ecuador, Peru (1)', 'Eritrea, Ethiopia (1)', 

---
## Section 6 — Cross-Source Coverage Audit

Build an `iso3 × source` matrix where each cell contains the row count for that
country in that source. This reveals coverage gaps before panel construction.

**Known expected gaps** (not data errors):
- `TWN` (Taiwan): no World Bank WDI entry (political exclusion)
- `PSE` (Palestine): WDI from 1994 only
- `XKX` (Kosovo): pre-2008 rows attributed to `SRB` in most sources
- `SSD` (South Sudan): pre-2011 rows attributed to `SDN`


In [21]:
_clean_sources = {
    "ucdp_acd":       (ucdp_acd_clean,       "iso3"),
    "ucdp_ged":       (ucdp_ged_clean,        "iso3"),
    "ucdp_brd":       (ucdp_brd_clean,        "iso3"),
    "ucdp_dyadic":    (ucdp_dyadic_clean,     "iso3"),
    "sipri_milex":    (sipri_milex_clean,      "iso3"),
    "sipri_tiv":      (sipri_tiv_register_clean, "recipient_iso3"),
    "worldbank_wdi":  (worldbank_wdi_clean,    "iso3"),
    "vdem":           (vdem_clean,             "iso3"),
    "cow_cinc":       (cow_cinc_clean,         "iso3"),
}

matrix_rows = {}
for src, (df, col) in _clean_sources.items():
    matrix_rows[src] = df[col].value_counts()

coverage = pd.DataFrame(matrix_rows).fillna(0).astype(int)
coverage.index.name = "iso3"
coverage = coverage.sort_index()

# Save
(CLEAN_DIR).mkdir(exist_ok=True)
coverage.to_csv(CLEAN_DIR / "_iso3_coverage_matrix.csv")
print(f"Coverage matrix: {coverage.shape[0]} countries × {coverage.shape[1]} sources")

# Countries with full coverage (non-zero in ALL sources)
full_cov = coverage[(coverage > 0).all(axis=1)]
print(f"Countries with full coverage across all sources: {len(full_cov)}")

# UCDP conflict countries missing WDI GDP
ucdp_iso3s = set(ucdp_acd_clean["iso3"].dropna().unique())
wdi_iso3s  = set(worldbank_wdi_clean["iso3"].dropna().unique())
missing_wdi = sorted(ucdp_iso3s - wdi_iso3s)
print(f"\nCountries in UCDP ACD but missing WDI GDP: {missing_wdi}")

# Known expected gaps
for code, note in [("TWN", "Taiwan: excluded from WDI (political)"),
                   ("PSE", "Palestine: WDI from 1994 only"),
                   ("XKX", "Kosovo: pre-2008 data attributed to SRB"),
                   ("SSD", "South Sudan: pre-2011 data attributed to SDN")]:
    wdi_rows = (worldbank_wdi_clean["iso3"] == code).sum()
    print(f"  {code}: {wdi_rows} WDI rows  [{note}]")


Coverage matrix: 213 countries × 9 sources
Countries with full coverage across all sources: 82

Countries in UCDP ACD but missing WDI GDP: ['KOR', 'PRK', 'SOM', 'TWN', 'VEN', 'YEM']
  TWN: 0 WDI rows  [Taiwan: excluded from WDI (political)]
  PSE: 35 WDI rows  [Palestine: WDI from 1994 only]
  XKX: 65 WDI rows  [Kosovo: pre-2008 data attributed to SRB]
  SSD: 65 WDI rows  [South Sudan: pre-2011 data attributed to SDN]


---
## Section 7 — Save Unmatched Audit

All country names that the resolver could not map to an ISO3 code are logged
during the run. Reviewing this file manually is the final QA step before NB-03.


In [22]:
unmatched_df = resolver.report_unmatched()
unmatched_df.to_csv(CLEAN_DIR / "_unmatched_audit.csv", index=False)

print(f"Unmatched names logged: {len(unmatched_df)}")
if len(unmatched_df) > 0:
    print(unmatched_df.head(20).to_string(index=False))

print("\n" + "="*60)
print("Review data/clean/_unmatched_audit.csv manually.")
print("Add any legitimate country names to GLOBAL_OVERRIDES or")
print("dataset_overrides, then re-run NB02.")
print("="*60)


Unmatched names logged: 702
 dataset                                                      name  count
cow_cinc                                                       BAD      1
cow_cinc                                                       BAV      1
cow_cinc                                                       HAN      1
cow_cinc                                                       HSE      1
cow_cinc                                                       HSG      1
cow_cinc                                                       MEC      1
cow_cinc                                                       MOD      1
cow_cinc                                                       PAP      1
cow_cinc                                                       SAX      1
cow_cinc                                                       SIC      1
cow_cinc                                                       SNM      1
cow_cinc                                                       TUS      1
cow_cinc  

---
## NB-02 Complete

All clean Parquet files written to `data/clean/`:

| File | Description |
|------|-------------|
| `ucdp_acd_clean.parquet` | Armed conflicts, location-exploded |
| `ucdp_ged_clean.parquet` | Georeferenced events with year_month |
| `ucdp_brd_clean.parquet` | Battle-related deaths |
| `ucdp_dyadic_clean.parquet` | Dyadic conflicts with actor ISO3 lists |
| `ucdp_participation_long.parquet` | Long-form participation with `is_extraterritorial` |
| `ucdp_participation_by_country_year.parquet` | Aggregated participation |
| `sipri_milex_clean.parquet` | Military expenditure |
| `sipri_tiv_register_clean.parquet` | Arms transfer register (state-to-state) |
| `sipri_tiv_non_state.parquet` | Non-state recipient transfers |
| `sipri_tiv_by_country_year_clean.parquet` | TIV totals by recipient-year |
| `sipri_tiv_by_country_year_category_clean.parquet` | TIV by recipient-year-category |
| `worldbank_wdi_clean.parquet` | WDI (aggregates dropped) |
| `vdem_clean.parquet` | V-Dem Electoral Democracy Index |
| `cow_cinc_clean.parquet` | COW CINC scores ≤2016 |
| `_unmatched_audit.csv` | Names that failed ISO3 resolution |
| `_iso3_coverage_matrix.csv` | iso3 × source row count matrix |
